# Isolation Forest

Ensemble of small random trees; anomalies are points that take fewer random splits to isolate than normal points. Small forest kept here on purpose (fewer trees -> smaller export).

Trained STRICTLY on `normal` sessions only — motor_off and transition sessions are excluded entirely, so this model's job is simply: does this look like normal running, or not.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib skl2onnx onnx

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnx

In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s", "PWM", "Irms", "Vrms",
    "Current_Variance", "Current_Crest_Factor",
    "Vib_RMS", "Vib_Min", "Vib_Q1", "Vib_Median",
    "Vib_Q3", "Vib_Max", "Vib_Variance", "Vib_Skewness",
]

N_FEATURES = len(FEATURE_COLUMNS)
WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
MAX_SAMPLES = "auto"
MAX_FEATURES = 1.0
ANOMALY_SCORE_PERCENTILE = 99.0
FILE_ANOMALY_FRACTION_THRESHOLD = 0.10

OUTPUT_DIR = "pure_normal_isolation_forest_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
FINAL_EVAL_DIR = os.path.join(OUTPUT_DIR, "final_model_evaluation")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, FINAL_EVAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Features:", N_FEATURES)
print("Window shape:", (WINDOW_LEN, N_FEATURES))
print("Flattened input size:", WINDOW_LEN * N_FEATURES)

In [ ]:
# ==========================================================
# CELL 3 — normal and anomaly file detection
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def normal_state_for(path):
    p, stem = normalized_path(path), Path(path).stem.lower()
    excluded = ["motor_off", "motor_being_on", "motor_being_off", "transition", "off_on_spikes"]
    if any(word in p for word in excluded):
        return None
    return "normal_running" if "/normal/" in p or stem.startswith("normal_") or stem == "normal" else None


def anomaly_label_for(path):
    p = normalized_path(path)
    if "motor_being_on" in p:
        return "motor_being_on"
    if "motor_being_off" in p:
        return "motor_being_off"
    if "/motor_off/" in p or "motor_off" in Path(p).stem.lower():
        return "motor_off"
    if "electrical fault" in p or "abnormal_current" in p:
        return "electrical_fault"
    if "mechanical fault" in p or "abnormal_vibration" in p:
        return "mechanical_fault"
    if "thermal fault" in p or "abnormal_temp" in p:
        return "thermal_fault"
    return None

In [ ]:
# ==========================================================
# CELL 4 — discover and read files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(os.path.join(base_path, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
    return sorted(set(files))


def read_one(path):
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower() == ".xlsx" else pd.read_csv(path)
    missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")

    clean = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce") / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(f"{path.name} has {len(clean)} valid rows; need at least {WINDOW_LEN}.")
    return clean


files = discover_files(BASE_PATH)
if not files:
    raise RuntimeError(f"No CSV/XLSX files found under: {BASE_PATH}")

print("Total files discovered:", len(files))

In [ ]:
# ==========================================================
# CELL 5 — load only normal WITH-LOAD sessions for training
# ==========================================================
def is_normal_with_load(path):
    p = (
        normalized_path(path)
        .replace(" ", "_")
        .replace("-", "_")
    )

    # Reject without-load files first.
    without_load_markers = [
        "without_load",
        "withoutload",
        "no_load",
        "noload",
        "unloaded",
    ]

    if any(marker in p for marker in without_load_markers):
        return False

    # Accept with-load files.
    with_load_markers = [
        "with_load",
        "withload",
        "normal_load",
        "normal_loaded",
    ]

    return any(marker in p for marker in with_load_markers)


healthy_sessions = []
fault_sessions = []
excluded_normal_files = []

for file_path in files:
    normal_state = normal_state_for(file_path)
    fault_label = anomaly_label_for(file_path)

    if normal_state is None and fault_label is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(file_path, BASE_PATH),
        )
        continue

    # Only normal WITH-LOAD files may enter healthy_sessions.
    if (
        normal_state is not None
        and not is_normal_with_load(file_path)
    ):
        relative_path = os.path.relpath(
            file_path,
            BASE_PATH,
        )

        excluded_normal_files.append(relative_path)

        print(
            "EXCLUDED normal without load:",
            relative_path,
        )
        continue

    try:
        df = read_one(file_path)

    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    record = {
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(
            file_path,
            BASE_PATH,
        ),
        "rows": len(df),
    }

    if normal_state is not None:
        record.update({
            "state": normal_state,
            "load_condition": "with_load",
            "binary_label": 0,
        })

        healthy_sessions.append(record)

        print(
            f"N | {'normal_with_load':20s} | "
            f"{len(df):6d} | {record['file']}"
        )

    else:
        record.update({
            "fault_label": fault_label,
            "binary_label": 1,
        })

        fault_sessions.append(record)

        print(
            f"A | {fault_label:20s} | "
            f"{len(df):6d} | {record['file']}"
        )


if not healthy_sessions:
    raise RuntimeError(
        "No normal WITH-LOAD files were loaded. "
        "Check the filenames or folder names."
    )

if len(healthy_sessions) < N_SPLITS:
    raise RuntimeError(
        f"Only {len(healthy_sessions)} normal with-load files "
        f"were loaded, but N_SPLITS={N_SPLITS}."
    )

if not fault_sessions:
    raise RuntimeError(
        "No anomaly files were loaded."
    )


print("\nNORMAL WITH-LOAD FILES USED FOR TRAINING")
for session in healthy_sessions:
    print("  USED:", session["file"])


print("\nNORMAL WITHOUT-LOAD FILES EXCLUDED")
for file_name in excluded_normal_files:
    print("  EXCLUDED:", file_name)


print("\nDATASET SUMMARY")
print("Normal with-load training files:", len(healthy_sessions))
print("Excluded normal files:", len(excluded_normal_files))
print("Fault evaluation files:", len(fault_sessions))

print("\nFAULT FILE DISTRIBUTION")
print(
    pd.Series([
        session["fault_label"]
        for session in fault_sessions
    ]).value_counts()
)

In [ ]:
# ==========================================================
# CELL 6 — window helpers
# ==========================================================
def make_windows_from_array(X_raw, scaler):
    X = scaler.transform(X_raw).astype(np.float32)
    starts = np.arange(0, len(X) - WINDOW_LEN + 1, WINDOW_STEP, dtype=np.int64)
    if len(starts) == 0:
        return None, None
    windows = np.stack([X[i:i + WINDOW_LEN] for i in starts]).astype(np.float32)
    return windows, starts


def build_windows_from_sessions(sessions, scaler):
    windows = []
    for session in sessions:
        Xw, _ = make_windows_from_array(session["X"], scaler)
        if Xw is not None:
            windows.append(Xw)
    if not windows:
        return np.empty((0, WINDOW_LEN, N_FEATURES), dtype=np.float32)
    return np.concatenate(windows)


def flatten_windows(X):
    return X.reshape(len(X), -1)



In [ ]:
# ==========================================================
# CELL 7 — Isolation Forest model and scoring
# ==========================================================
def build_isolation_forest():
    return IsolationForest(
        n_estimators=50,
        max_samples=MAX_SAMPLES,
        max_features=MAX_FEATURES,
        contamination="auto",
        random_state=SEED,
        n_jobs=-1,
    )


def anomaly_score(model, X):
    return (-model.decision_function(flatten_windows(X))).astype(np.float32)


def calibrate_threshold(model, X_normal):
    scores = anomaly_score(model, X_normal)
    return float(np.percentile(scores, ANOMALY_SCORE_PERCENTILE))


In [ ]:
# ==========================================================
# CELL 8 — file-level and window-level predictions
# ==========================================================
def aggregate_predictions(sessions, model, scaler, threshold):
    file_rows, window_rows = [], []

    for session in sessions:
        Xw, starts = make_windows_from_array(session["X"], scaler)
        if Xw is None:
            continue

        scores = anomaly_score(model, Xw)
        flags = (scores > threshold).astype(np.int64)
        state = session.get("state", session.get("fault_label", "unknown"))
        true_label = int(session["binary_label"])
        anomaly_fraction = float(np.mean(flags))
        file_prediction = int(anomaly_fraction >= FILE_ANOMALY_FRACTION_THRESHOLD)

        file_rows.append({
            "file": session["file"],
            "state": state,
            "true_label": true_label,
            "pred_label": file_prediction,
            "windows": len(Xw),
            "anomaly_windows": int(flags.sum()),
            "anomaly_fraction": anomaly_fraction,
            "mean_score": float(scores.mean()),
            "max_score": float(scores.max()),
            "score_threshold": threshold,
        })

        for i, start in enumerate(starts):
            window_rows.append({
                "file": session["file"],
                "state": state,
                "window": i,
                "start_row": int(start),
                "true_label": true_label,
                "pred_label": int(flags[i]),
                "anomaly_score": float(scores[i]),
                "score_threshold": threshold,
            })

    return pd.DataFrame(file_rows), pd.DataFrame(window_rows)


In [ ]:
# ==========================================================
# CELL 9 — build cross-validation table
# ==========================================================
all_sessions = healthy_sessions + fault_sessions

cv_table = pd.DataFrame({
    "index": np.arange(len(all_sessions)),
    "label": [s["binary_label"] for s in all_sessions],
    "group": [s["file"] for s in all_sessions],
})

print(cv_table["label"].value_counts())


In [ ]:
# ==========================================================
# CELL 10 — five-fold file-wise cross-validation
# ==========================================================
splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

cv_metrics, cv_file_results, cv_window_results, cv_test_files = [], [], [], []

for fold, (train_idx, test_idx) in enumerate(
    splitter.split(cv_table, cv_table["label"], groups=cv_table["group"]), start=1
):
    print("\n" + "=" * 70)
    print(f"CV FOLD {fold}/{N_SPLITS}")
    print("=" * 70)

    train_sessions = [all_sessions[i] for i in train_idx]
    test_sessions = [all_sessions[i] for i in test_idx]
    normal_train = [s for s in train_sessions if s["binary_label"] == 0]

    rng = np.random.default_rng(SEED + fold)
    order = rng.permutation(len(normal_train))
    n_cal = max(1, round(0.20 * len(order)))
    cal_ids = set(order[:n_cal].tolist())

    fit_sessions = [s for i, s in enumerate(normal_train) if i not in cal_ids]
    cal_sessions = [s for i, s in enumerate(normal_train) if i in cal_ids]

    scaler = MinMaxScaler().fit(np.concatenate([s["X"] for s in fit_sessions]))
    X_fit = build_windows_from_sessions(fit_sessions, scaler)
    X_cal = build_windows_from_sessions(cal_sessions, scaler)

    model = build_isolation_forest()
    model.fit(flatten_windows(X_fit))
    threshold = calibrate_threshold(model, X_cal)

    file_df, window_df = aggregate_predictions(test_sessions, model, scaler, threshold)
    file_df.insert(0, "fold", fold)
    window_df.insert(0, "fold", fold)

    cv_file_results.append(file_df)
    cv_window_results.append(window_df)

    for s in test_sessions:
        state = s.get("state", s.get("fault_label", "unknown"))
        cv_test_files.append({
            "fold": fold, "file": s["file"], "state": state,
            "true_label": s["binary_label"],
        })
        print(f"label={s['binary_label']} | state={state} | file={s['file']}")

    y_true = window_df["true_label"].to_numpy()
    y_pred = window_df["pred_label"].to_numpy()
    y_score = window_df["anomaly_score"].to_numpy()

    result = {
        "fold": fold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        "score_threshold": threshold,
    }

    cv_metrics.append(result)
    print("\nWINDOW-LEVEL FOLD RESULTS")
    print(pd.Series(result))
    print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))


In [ ]:
# ==========================================================
# CELL 11 — combined cross-validation results
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_test_files_df = pd.DataFrame(cv_test_files)
cv_file_results_df = pd.concat(cv_file_results, ignore_index=True)
cv_window_results_df = pd.concat(cv_window_results, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_test_files_df.to_csv(os.path.join(CV_DIR, "cv_test_files.csv"), index=False)
cv_file_results_df.to_csv(os.path.join(CV_DIR, "cv_file_predictions.csv"), index=False)
cv_window_results_df.to_csv(os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False)

y_true = cv_window_results_df["true_label"].to_numpy()
y_pred = cv_window_results_df["pred_label"].to_numpy()
y_score = cv_window_results_df["anomaly_score"].to_numpy()

overall_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_score),
}

print("\nOVERALL CV WINDOW-LEVEL RESULTS")
print(pd.Series(overall_metrics))
print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))

pd.DataFrame([overall_metrics]).to_csv(os.path.join(CV_DIR, "overall_metrics.csv"), index=False)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["normal window", "anomaly window"]).plot(values_format="d")
plt.title("Combined CV Window-Level Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)
plt.show()


In [ ]:
# ==========================================================
# CELL 12 — train final Isolation Forest on 100% normal data
# ==========================================================
final_scaler = MinMaxScaler().fit(np.concatenate([s["X"] for s in healthy_sessions]))
X_all_healthy = build_windows_from_sessions(healthy_sessions, final_scaler)

final_model = build_isolation_forest()
final_model.fit(flatten_windows(X_all_healthy))

FINAL_SCORE_THRESHOLD = calibrate_threshold(final_model, X_all_healthy)

print("\nFINAL SCORE THRESHOLD:", FINAL_SCORE_THRESHOLD)
print("All pure-normal windows:", X_all_healthy.shape)



In [ ]:
# ==========================================================
# CELL 13 — final model evaluation
# ==========================================================
rng = np.random.default_rng(SEED)

n_normal_eval = max(1, round(0.20 * len(healthy_sessions)))
normal_ids = rng.choice(len(healthy_sessions), size=n_normal_eval, replace=False)
selected_normal = [healthy_sessions[int(i)] for i in normal_ids]

fault_groups = {}
for session in fault_sessions:
    fault_groups.setdefault(session["fault_label"], []).append(session)

selected_faults = [group[int(rng.integers(len(group)))] for group in fault_groups.values()]
evaluation_sessions = selected_normal + selected_faults

print("\nFINAL EVALUATION FILES")
for session in evaluation_sessions:
    state = session.get("state", session.get("fault_label", "unknown"))
    print(f"label={session['binary_label']} | state={state} | file={session['file']}")

eval_file_df, eval_window_df = aggregate_predictions(
    evaluation_sessions, final_model, final_scaler, FINAL_SCORE_THRESHOLD
)

y_true = eval_window_df["true_label"].to_numpy()
y_pred = eval_window_df["pred_label"].to_numpy()
y_score = eval_window_df["anomaly_score"].to_numpy()

final_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "normal_accuracy": np.mean(y_pred[y_true == 0] == 0),
    "anomaly_accuracy": np.mean(y_pred[y_true == 1] == 1),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_score),
    "score_threshold": FINAL_SCORE_THRESHOLD,
}

print("\nFINAL WINDOW-LEVEL RESULTS")
print(pd.Series(final_metrics))
print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], zero_division=0))

eval_file_df.to_csv(os.path.join(FINAL_EVAL_DIR, "file_predictions.csv"), index=False)
eval_window_df.to_csv(os.path.join(FINAL_EVAL_DIR, "window_predictions.csv"), index=False)
pd.DataFrame([final_metrics]).to_csv(os.path.join(FINAL_EVAL_DIR, "metrics.csv"), index=False)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["normal window", "anomaly window"]).plot(values_format="d")
plt.title("Final Isolation Forest Window-Level Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(FINAL_EVAL_DIR, "confusion_matrix.png"), dpi=180)
plt.show()


In [ ]:
# ==========================================================
# CELL 14 — save model, scaler, threshold, and metadata
# ==========================================================
MODEL_PATH = os.path.join(FINAL_DIR, "pure_normal_isolation_forest.pkl")
SCALER_PATH = os.path.join(FINAL_DIR, "pure_normal_scaler.pkl")
THRESHOLD_PATH = os.path.join(FINAL_DIR, "isolation_forest_threshold.npz")
METADATA_PATH = os.path.join(FINAL_DIR, "model_metadata.csv")

joblib.dump(final_model, MODEL_PATH)
joblib.dump(final_scaler, SCALER_PATH)
np.savez(
    THRESHOLD_PATH,
    score_threshold=np.float32(FINAL_SCORE_THRESHOLD),
    file_anomaly_fraction_threshold=np.float32(FILE_ANOMALY_FRACTION_THRESHOLD),
)

pd.DataFrame({
    "feature_id": np.arange(N_FEATURES),
    "feature_name": FEATURE_COLUMNS,
    "scaler_min": final_scaler.data_min_,
    "scaler_max": final_scaler.data_max_,
}).to_csv(METADATA_PATH, index=False)

ONNX_PATH = os.path.join(
    FINAL_DIR,
    "pure_normal_isolation_forest.onnx",
)

# Isolation Forest receives one flattened window:
# 30 rows × 14 features = 420 inputs.
input_size = WINDOW_LEN * N_FEATURES

initial_type = [
    (
        "float_input",
        FloatTensorType([None, input_size]),
    )
]

onnx_model = convert_sklearn(
    final_model,
    initial_types=initial_type,
    target_opset={
        "":15,
        "ai.onnx.ml": 3,
    },
)

onnx.save_model(
    onnx_model,
    ONNX_PATH,
)


print("Saved:", MODEL_PATH)
print("Saved:", SCALER_PATH)
print("Saved:", THRESHOLD_PATH)
print("Saved:", METADATA_PATH)
print("Saved ONNX model:", ONNX_PATH)
print(
    "ONNX model size KB:",
    os.path.getsize(ONNX_PATH) / 1024,
)

In [ ]:
# ==========================================================
# CELL 14.1 — export complete Isolation Forest as C header
# ==========================================================
import math

H=os.path.join(FINAL_DIR,"isolation_forest_model.h")
O=[0]; L=[]; R=[]; F=[]; T=[]; A=[]

def c(n):
    if n<=1:return 0.
    if n==2:return 1.
    return 2*(math.log(n-1)+0.5772156649)-2*(n-1)/n

for e in final_model.estimators_:
    t=e.tree_; L+=t.children_left.tolist(); R+=t.children_right.tolist()
    F+=t.feature.tolist(); T+=t.threshold.tolist()
    A+=[c(int(t.n_node_samples[i])) if t.children_left[i]==t.children_right[i] else 0. for i in range(t.node_count)]
    O.append(O[-1]+t.node_count)

def w(f,n,x,typ="float"):
    s=",".join(str(int(v)) if typ=="int" else f"{float(v):.9g}f" for v in x)
    f.write(f"static const {'int' if typ=='int' else 'float'} {n}[{len(x)}]={{{s}}};\n")

with open(H,"w") as f:
    f.write("#ifndef IF_MODEL_H\n#define IF_MODEL_H\n#include <math.h>\n")
    f.write(f"#define NF {N_FEATURES}\n#define WL {WINDOW_LEN}\n#define NI {WINDOW_LEN*N_FEATURES}\n#define NT {len(final_model.estimators_)}\n")
    f.write(f"#define DIV {SCALE_DIVISOR:.9g}f\n#define CM {c(int(final_model.max_samples_)):.9g}f\n#define OFF {float(final_model.offset_):.9g}f\n#define ST {FINAL_SCORE_THRESHOLD:.9g}f\n")
    w(f,"MN",final_scaler.data_min_); w(f,"MX",final_scaler.data_max_)
    w(f,"O",O,"int"); w(f,"L",L,"int"); w(f,"R",R,"int"); w(f,"F",F,"int")
    w(f,"T",T); w(f,"A",A)
    f.write(r'''
static inline float if_path(const float*x,int q){int b=O[q],n=0;float d=0;for(;;){int i=b+n;if(L[i]==R[i])return d+A[i];n=x[F[i]]<=T[i]?L[i]:R[i];d++;}}
static inline int isolation_forest(const float raw[WL][NF],float*score){
 float x[NI],p=0;for(int t=0;t<WL;t++)for(int j=0;j<NF;j++){float v=raw[t][j]/DIV,d=MX[j]-MN[j];x[t*NF+j]=d?((v-MN[j])/d):0;}
 for(int q=0;q<NT;q++)p+=if_path(x,q);float ss=-powf(2.f,-(p/NT)/CM);*score=-(ss-OFF);return *score>ST;}
#endif
''')
print("Saved:",H)
print("Header KB:",round(os.path.getsize(H)/1024,2))

In [ ]:
# ==========================================================
# CELL 14.2 — export ONNX preprocessing parameters as .h file
# ==========================================================
PARAM_HEADER_PATH = os.path.join(
    FINAL_DIR,
    "isolation_forest_params.h",
)

def format_c_float(value):
    """Convert a Python/NumPy number into a valid C float."""
    value = float(value)

    if not np.isfinite(value):
        raise ValueError(f"Cannot export non-finite value: {value}")

    return f"{value:.9g}f"


def write_c_float_array(file, name, values):
    values = np.asarray(values, dtype=np.float32).reshape(-1)

    file.write(
        f"static const float {name}[{len(values)}] = {{\n"
    )

    for start in range(0, len(values), 4):
        row = values[start:start + 4]
        text = ", ".join(format_c_float(v) for v in row)

        if start + 4 < len(values):
            text += ","

        file.write(f"    {text}\n")

    file.write("};\n\n")


with open(PARAM_HEADER_PATH, "w", encoding="utf-8") as file:
    file.write(
        "#ifndef ISOLATION_FOREST_PARAMS_H\n"
        "#define ISOLATION_FOREST_PARAMS_H\n\n"
    )

    file.write("#include <stdint.h>\n\n")

    # Model dimensions
    file.write(f"#define IF_WINDOW_LEN {WINDOW_LEN}\n")
    file.write(f"#define IF_N_FEATURES {N_FEATURES}\n")
    file.write(
        f"#define IF_INPUT_SIZE "
        f"(IF_WINDOW_LEN * IF_N_FEATURES)\n\n"
    )

    # Timing and window movement
    file.write(f"#define IF_WINDOW_STEP {WINDOW_STEP}\n")
    file.write(
        f"#define IF_ROW_PERIOD_SEC "
        f"{format_c_float(ROW_PERIOD_SEC)}\n\n"
    )

    # Scaling and thresholds
    file.write(
        f"#define IF_SCALE_DIVISOR "
        f"{format_c_float(SCALE_DIVISOR)}\n"
    )

    file.write(
        f"#define IF_ANOMALY_SCORE_THRESHOLD "
        f"{format_c_float(FINAL_SCORE_THRESHOLD)}\n"
    )

    file.write(
        f"#define IF_FILE_ANOMALY_FRACTION_THRESHOLD "
        f"{format_c_float(FILE_ANOMALY_FRACTION_THRESHOLD)}\n\n"
    )

    # Feature indices
    for index, feature_name in enumerate(FEATURE_COLUMNS):
        macro_name = (
            feature_name
            .upper()
            .replace(" ", "_")
            .replace("-", "_")
        )

        file.write(
            f"#define IF_FEATURE_{macro_name} {index}\n"
        )

    file.write("\n")

    # MinMaxScaler parameters
    write_c_float_array(
        file,
        "IF_SCALER_DATA_MIN",
        final_scaler.data_min_,
    )

    write_c_float_array(
        file,
        "IF_SCALER_DATA_MAX",
        final_scaler.data_max_,
    )

    # Function used before feeding data into the ONNX model
    file.write(
r"""/*
 * Input:
 *     raw_window[30][14]
 *
 * Output:
 *     model_input[420]
 *
 * Flattening order:
 *     row 0 feature 0 ... feature 13,
 *     row 1 feature 0 ... feature 13,
 *     ...
 *     row 29 feature 0 ... feature 13.
 *
 * This preprocessing must be identical to Python:
 *
 *     divided_value = raw_value / SCALE_DIVISOR
 *     scaled_value =
 *         (divided_value - scaler_min) /
 *         (scaler_max - scaler_min)
 */
static inline void isolation_forest_prepare_input(
    const float raw_window[IF_WINDOW_LEN][IF_N_FEATURES],
    float model_input[IF_INPUT_SIZE])
{
    uint32_t input_index = 0U;

    for (uint32_t row = 0U; row < IF_WINDOW_LEN; row++)
    {
        for (uint32_t feature = 0U;
             feature < IF_N_FEATURES;
             feature++)
        {
            const float divided_value =
                raw_window[row][feature] /
                IF_SCALE_DIVISOR;

            const float minimum =
                IF_SCALER_DATA_MIN[feature];

            const float maximum =
                IF_SCALER_DATA_MAX[feature];

            const float range = maximum - minimum;

            if (range != 0.0f)
            {
                model_input[input_index] =
                    (divided_value - minimum) / range;
            }
            else
            {
                model_input[input_index] = 0.0f;
            }

            input_index++;
        }
    }
}

/*
 * The Python anomaly score is:
 *
 *     anomaly_score = -model.decision_function(X)
 *
 * Therefore, if the ONNX score output represents
 * decision_function(X), negate it before comparing.
 */
static inline uint8_t isolation_forest_is_anomaly(
    float onnx_decision_score)
{
    const float anomaly_score = -onnx_decision_score;

    return (anomaly_score >
            IF_ANOMALY_SCORE_THRESHOLD) ? 1U : 0U;
}

#endif /* ISOLATION_FOREST_PARAMS_H */
"""
    )


print("Saved parameter header:", PARAM_HEADER_PATH)
print(
    "Header size:",
    round(os.path.getsize(PARAM_HEADER_PATH) / 1024, 2),
    "KB",
)

print("\nParameters exported:")
print("Input shape:", (1, WINDOW_LEN * N_FEATURES))
print("Score threshold:", FINAL_SCORE_THRESHOLD)
print(
    "File anomaly-fraction threshold:",
    FILE_ANOMALY_FRACTION_THRESHOLD,
)

# Display a clickable link in Jupyter Notebook
try:
    from IPython.display import FileLink, display
    display(FileLink(PARAM_HEADER_PATH))
except Exception:
    pass

# Automatically download when running in Google Colab
try:
    from google.colab import files as colab_files
    colab_files.download(PARAM_HEADER_PATH)
except ImportError:
    print(
        "\nNot running in Google Colab. "
        "Use the clickable link above or open this folder:\n",
        os.path.abspath(FINAL_DIR),
    )

In [ ]:
# ==========================================================
# CELL 15 — manual test function
# ==========================================================
def manual_isolation_forest_prediction(test_file):
    model = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    threshold_data = np.load(THRESHOLD_PATH)

    threshold = float(threshold_data["score_threshold"])
    file_threshold = float(threshold_data["file_anomaly_fraction_threshold"])

    df = read_one(test_file)
    X_test, starts = make_windows_from_array(df.to_numpy(dtype=np.float32), scaler)
    scores = anomaly_score(model, X_test)
    flags = (scores > threshold).astype(np.int64)

    anomaly_fraction = float(np.mean(flags))
    file_decision = "ANOMALY" if anomaly_fraction >= file_threshold else "HEALTHY"

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC,
        "anomaly_score": scores,
        "score_threshold": threshold,
        "is_anomaly": flags,
    })

    stem = Path(test_file).stem
    output_csv = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}.csv")
    result.to_csv(output_csv, index=False)

    print("\nMANUAL ISOLATION FOREST TEST")
    print("File:", test_file)
    print("Windows:", len(result))
    print("Anomalous windows:", int(flags.sum()))
    print("Anomaly fraction:", anomaly_fraction)
    print("File decision:", file_decision)
    print("Score threshold:", threshold)
    print("Saved CSV:", output_csv)

    plt.figure(figsize=(12, 4))
    plt.plot(result["time_sec"], result["anomaly_score"])
    plt.axhline(threshold, linestyle="--", label="score threshold")
    plt.xlabel("Time [s]")
    plt.ylabel("Isolation Forest anomaly score")
    plt.title(f"Isolation Forest Manual Test — {stem}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_score.png"), dpi=150)
    plt.show()

    return result

In [ ]:
# ==========================================================
# CELL 16 — select manual test file
# ==========================================================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor CSV or Excel file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_isolation_forest_prediction(TEST_FILE)
        print(manual_result.head(30))
    else:
        print("No file selected.")